# J2S2 — Scoring crédit & sélection de variables (LR + RF) · Enrichi Audit Tuffery
## BankRisk Intelligence Platform · Jour 2 · Session 2

**Objectif :** Construire un score de crédit de façon **méthodique** — de l'ensemble des variables candidates jusqu'au modèle final — en maîtrisant la **sélection de variables** (forward, backward, RFE, Lasso).

**Entrée :** `credit_risk_kmeans.parquet` (32 581 lignes, produit par J2S1).
**Livrable :** `credit_risk_scored.parquet`

> **v2 — Enrichi suite audit Tuffery Ch.18 :** reject inference documenté, holdout set, grille de points 0-100, courbe ROC, tranches de risque, score stratégique/opérationnel. enrichi de `rf_score_proba`, `rf_decision_042`, `lr_shadow_score`.

---

### Ce que vous allez apprendre

1. **Ce qu'est un score de crédit** — probabilité de défaut, scorecard, odds-ratios, et pourquoi la régression logistique reste la référence réglementaire.
2. **L'analyse bivariée typée** — choisir la bonne mesure d'association selon le type des variables (Spearman, V de Cramér, rapport de corrélation η) + diagnostic de multicolinéarité (VIF).
3. **La sélection de variables** — forward, backward, RFE, Lasso : quatre méthodes pour ne garder que les variables utiles.
4. **Deux modèles, deux philosophies** — le Random Forest performant (production, expliqué par SHAP en J2S3) et le scorecard logistique interprétable (benchmark réglementaire).

---

### Position dans la chaîne Parquet BankRisk
```
credit_risk_dataset.csv          (12 cols — CSV Kaggle CC0)
    → drop(loan_grade)           (variable pré-octroi, anti-leakage)
        → credit_features_j1.parquet    (J1S2)
            → credit_risk_clean.parquet (J1S4)
                → credit_risk_kmeans.parquet ← ENTRÉE J2S2 (J2S1, +cluster_id)
                    → credit_risk_scored.parquet ← LIVRABLE J2S2 (+scores RF/LR)
                        → SHAP explicabilité (J2S3)
```

*Dataset : credit_risk_dataset.csv — 32 581 lignes · CC0 · Taux de défaut : 21,8 %*


---
## Bloc 0 — Setup · ROOT + pip install + imports (cellule unique)

> **Règle absolue :** `ROOT` est défini **dans** cette cellule — ne jamais la diviser.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 0 — Setup · ROOT detection + pip install + imports (cellule unique)
# ═══════════════════════════════════════════════════════════════════════════
import sys, os
from pathlib import Path

# Détection environnement : Google Colab ou VS Code local
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/bankrisk')
except ImportError:
    IN_COLAB = False
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'data').exists() or (ROOT / 'requirements.txt').exists():
            break
        ROOT = ROOT.parent

print(f"Environnement : {'Google Colab' if IN_COLAB else 'VS Code local'}")
print(f"ROOT : {ROOT}")

# Installation des dépendances
req_file = ROOT / 'requirements.txt'
if req_file.exists():
    os.system(f'{sys.executable} -m pip install -r {req_file} -q')
else:
    os.system(f'{sys.executable} -m pip install '
              'pandas numpy scikit-learn imbalanced-learn statsmodels scipy plotly pyarrow -q')

# ── Imports ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly
import plotly.express as px
import plotly.graph_objects as go

from scipy import stats
from scipy.stats import chi2_contingency, spearmanr

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector, RFE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score
)
from imblearn.over_sampling import SMOTENC
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"plotly  : {plotly.__version__}")   # plotly.__version__, pas px.__version__
print(f"sklearn : {__import__('sklearn').__version__}")

# Chemins Parquet (toujours ROOT / 'data' / 'processed' / 'fichier.parquet')
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

PARQUET_IN  = DATA_PROCESSED / 'credit_risk_kmeans.parquet'
PARQUET_OUT = DATA_PROCESSED / 'credit_risk_scored.parquet'

# Palette Ocean Executive BankRisk
NAVY   = '#021B2E'
DEEP   = '#065A82'
TEAL   = '#1C7293'
MINT   = '#02C39A'
ORANGE = '#FFA07A'

RANDOM_STATE = 42

print("\n✓ Setup complet — J2S2 prêt")


---
## Bloc 1 — Chargement & assertions qualité

> **Pré-requis :** `credit_risk_kmeans.parquet` produit par J2S1.
> Ce notebook est **autonome** : si le parquet est absent, il reconstruit le minimum nécessaire depuis le CSV.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 1 — Chargement et assertions qualité
# ═══════════════════════════════════════════════════════════════════════════

if not PARQUET_IN.exists():
    print("credit_risk_kmeans.parquet introuvable — reconstruction minimale depuis CSV...")
    csv_path = ROOT / 'data' / 'raw' / 'credit_risk_dataset.csv'
    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV source introuvable : {csv_path}\n"
            "Placez credit_risk_dataset.csv dans data/raw/ ou executez J1S2->J2S1 d'abord."
        )
    from sklearn.impute import SimpleImputer
    from sklearn.cluster import KMeans

    df_raw = pd.read_csv(csv_path)
    df_raw = df_raw.drop(columns=['loan_grade'], errors='ignore')

    df_raw['person_age']        = df_raw['person_age'].clip(upper=50)
    df_raw['person_emp_length'] = df_raw['person_emp_length'].clip(upper=18)
    df_raw['person_income']     = df_raw['person_income'].clip(upper=225200)

    imp = SimpleImputer(strategy='median')
    df_raw[['loan_int_rate', 'person_emp_length']] = imp.fit_transform(
        df_raw[['loan_int_rate', 'person_emp_length']]
    )

    df_raw['default_enc']   = (df_raw['cb_person_default_on_file'] == 'Y').astype(int)
    df_raw['home_RENT']     = (df_raw['person_home_ownership'] == 'RENT').astype(int)
    df_raw['home_MORTGAGE'] = (df_raw['person_home_ownership'] == 'MORTGAGE').astype(int)
    df_raw['home_OWN']      = (df_raw['person_home_ownership'] == 'OWN').astype(int)

    df_raw['debt_service_rate']     = df_raw['loan_int_rate'] * df_raw['loan_percent_income']
    df_raw['monthly_payment_proxy'] = df_raw['loan_amnt'] / (df_raw['person_income'] / 12)
    df_raw['log_income']            = np.log1p(df_raw['person_income'])
    df_raw['high_risk_intent']      = df_raw['loan_intent'].isin(
        ['DEBTCONSOLIDATION', 'MEDICAL']).astype(int)

    cf = ['loan_percent_income', 'loan_int_rate', 'monthly_payment_proxy',
          'debt_service_rate', 'person_income']
    sc = StandardScaler()
    Xs = sc.fit_transform(df_raw[cf])
    km = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42).fit(Xs)
    df_raw['cluster_id'] = km.labels_
    stats_c = df_raw.groupby('cluster_id')['loan_status'].mean().sort_values()
    mapping = {cid: lbl for cid, lbl in zip(
        stats_c.index, ['A_Faible_Risque', 'B_Risque_Modere', 'C_Risque_Eleve', 'D_Tres_Eleve'])}
    df_raw['cluster_label'] = df_raw['cluster_id'].map(mapping)

    df_raw.to_parquet(PARQUET_IN, index=False)
    print(f"  -> Reconstruit : {df_raw.shape}")

# ── Chargement ───────────────────────────────────────────────────────────────
df = pd.read_parquet(PARQUET_IN)
print(f"Dataset charge : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

# ── Assertions obligatoires ──────────────────────────────────────────────────
assert df.shape[0] == 32581, f"Shape inattendu : {df.shape[0]} lignes (attendu 32581)"
assert 'loan_grade' not in df.columns, "ERREUR : loan_grade presente — data leakage conceptuel"
assert df.isnull().sum().sum() == 0, f"ERREUR : {df.isnull().sum().sum()} NaN presents"

taux_defaut = df['loan_status'].mean() * 100
print(f"\n  Taux de defaut global : {taux_defaut:.1f} %  (reference : 21,8 %)")
print("\n✓ Assertions passees — J2S2 peut commencer")


---
## ⚠ Hypothèse critique — Reject Inference (Tuffery §18.3.3)

> **Audit Tuffery — Lacune #1 (critique)**

Ce modèle est entraîné **uniquement sur des emprunteurs ayant obtenu un prêt** (dataset Kaggle CC0).
Les dossiers qui auraient été *refusés* par un système de crédit existant sont **absents**.

### Pourquoi c'est un problème ?

Un système de scoring apprend à distinguer bons et mauvais payeurs **parmi les acceptés**.
Mais en production, il scorera aussi des profils jamais vus — potentiellement ceux qui auraient été refusés.
Le modèle peut se tromper systématiquement sur ces profils (**biais de sélection**).

Tuffery (section 18.3.3) appelle ce problème le **reject inference** et impose de reconstituer
une estimation du risque pour les refusés avant d'entraîner.

### Ce que nous faisons ici

Le dataset ne contenant pas de refusés, le reject inference ne peut pas être appliqué.
**Hypothèse de travail documentée :**

> *« Ce modèle est valide pour prédire le risque parmi une population similaire aux emprunteurs
> ayant déjà obtenu un prêt. Son pouvoir prédictif sur des profils systématiquement exclus
> d'un portefeuille existant n'est pas garanti et doit être validé avant déploiement. »*

**En production :** utiliser une méthode de reject inference (parcelling, augmentation par simulation,
ou mixture models) si des données de refusés sont disponibles.


---
## Bloc 2 — Qu'est-ce qu'un score de crédit ?

Un **score de crédit** transforme les caractéristiques d'un emprunteur en une **probabilité de défaut** (un nombre entre 0 et 1), puis en une **décision** (accorder / refuser) via un seuil.

### La régression logistique : le standard historique

Pendant des décennies, l'outil de référence du scoring bancaire a été la **régression logistique (LR)**. Pourquoi ?

- Elle produit une **équation lisible** : chaque variable a un **coefficient** dont on peut expliquer le sens à un comité de crédit ou à un régulateur.
- Ce coefficient se traduit en **odds-ratio** (rapport de cotes) : « à revenu égal, un emprunteur locataire a X fois plus de chances de faire défaut qu'un propriétaire ».
- On peut la convertir en **scorecard** : une grille de points par variable, auditable ligne à ligne.

C'est cette **transparence** qui explique sa longévité. La littérature le confirme : la régression logistique **reste le benchmark de l'industrie du risque crédit**, principalement parce que le manque d'interprétabilité des méthodes d'ensemble est jugé incompatible avec les exigences des régulateurs financiers *(Dumitrescu, Hué, Hurlin & Tokpavi, 2022, European Journal of Operational Research, vol. 297(3), pp. 1178-1192)*.

### Alors pourquoi utiliser un Random Forest ?

Parce qu'il est **plus performant**. Les études de référence montrent que les méthodes d'ensemble à base d'arbres, comme le **Random Forest**, offrent une capacité de classification supérieure à la régression logistique standard *(même source ; voir aussi Lessmann, Baesens, Seow & Thomas, 2015, EJOR vol. 247(1), pp. 124-136, qui compare de nombreux algorithmes sur plusieurs jeux de scoring)*.

### Deux modèles, deux philosophies — PAS l'un l'explication de l'autre

C'est le point conceptuel clé de cette session :

| | **Random Forest** | **Régression logistique** |
|---|---|---|
| Rôle | **Modèle de production** (performance) | **Benchmark interprétable** (réglementaire) |
| Force | AUC élevé (0,929) | Transparence, scorecard, odds-ratios |
| Explicabilité | via **SHAP** (J2S3) — sur *ses propres* prédictions | native (coefficients) |

> **Piège à éviter :** on n'utilise **pas** un LR pour « expliquer » les décisions d'un RF. Ce sont deux modèles différents qui produisent des scores différents. Expliquer un modèle par un autre donnerait une justification qui ne correspond pas à la décision réellement prise — un régulateur la rejetterait.
>
> La bonne pratique moderne (celle de ce programme) : le **RF est en production et on l'explique par SHAP** (ses propres contributions, J2S3). Le **scorecard LR est un livrable parallèle**, le modèle transparent de référence. Des travaux récents montrent d'ailleurs que les scores dérivés des valeurs de Shapley d'un modèle à base d'arbres atteignent une interprétabilité comparable à celle d'un scorecard logistique, tout en conservant une meilleure précision *(cf. framework SHAP pour scorecards, PMC11318906, 2024)*.

**Sources (liens) :**
- Dumitrescu et al. (2022), *EJOR* — https://ideas.repec.org/a/eee/ejores/v297y2022i3p1178-1192.html
- Comparaison RF vs LR en credit scoring — https://www.researchgate.net/publication/353326832_A_COMPARISON_OF_RANDOM_FOREST_AND_LOGISTIC_REGRESSION_MODEL_IN_CREDIT_SCORING_OF_RURAL_HOUSEHOLDS
- Framework interprétabilité SHAP pour scorecards — https://pmc.ncbi.nlm.nih.gov/articles/PMC11318906/


---
## Complément Bloc 2 — Score stratégique vs Score opérationnel (Tuffery §18.4 & §18.5)

> **Audit Tuffery — Recommandation #6**

Tuffery distingue deux usages du score qu'il faut documenter explicitement :

| | **Score stratégique** | **Score opérationnel** |
|---|---|---|
| **Usage** | Piloter le portefeuille | Décider dossier par dossier |
| **Fréquence** | Mensuel / trimestriel | Temps réel (à l'octroi) |
| **Acteur** | Direction des risques | Chargé de clientèle / API |
| **Sortie** | Taux de défaut attendu du portefeuille | Accord / Refus / Instruction |
| **Modèle J2S2** | `rf_score_proba` agrégé par segment | `rf_decision_042` (seuil 0,42) |

> Dans J2S2, `rf_score_proba` joue les deux rôles.
> En production, il faudrait des seuils et des règles métier distincts pour chaque usage.


---
## Bloc 3 — Variables candidates & analyse bivariée typée

Avant de sélectionner des variables, il faut mesurer **les associations** — entre elles, et avec la cible. Erreur fréquente : appliquer **Spearman partout**. Or Spearman ne vaut qu'entre variables **numériques**. Dès qu'une variable **catégorielle** entre en jeu, il faut une autre mesure.

### Règle de routage (à garder comme aide-mémoire)

| Type variable A | Type variable B | Mesure appropriée | Bornes |
|---|---|---|---|
| Numérique | Numérique | **Spearman** ρ | −1 à +1 |
| Catégorielle | Catégorielle | **V de Cramér** (via Khi²) | 0 à 1 |
| Numérique | Catégorielle | **Rapport de corrélation η** (eta) | 0 à 1 |

> On a introduit cette idée en **J1S3** (EDA). Ici on la **formalise** avec les mesures adaptées, plus un diagnostic de **multicolinéarité (VIF)**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 3.1 — Typage des variables candidates
# ═══════════════════════════════════════════════════════════════════════════

# Variables numériques continues
NUM_VARS = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
            'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']
NUM_VARS = [c for c in NUM_VARS if c in df.columns]

# Variables catégorielles nominales (forme brute, avant encodage binaire)
CAT_VARS = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']
CAT_VARS = [c for c in CAT_VARS if c in df.columns]

TARGET = 'loan_status'  # binaire

print("Typage des variables candidates :")
print(f"  Numériques ({len(NUM_VARS)}) : {NUM_VARS}")
print(f"  Catégorielles ({len(CAT_VARS)}) : {CAT_VARS}")
print(f"  Cible binaire : {TARGET}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 3.2 — Fonctions d'association typées
# ═══════════════════════════════════════════════════════════════════════════

def cramers_v_corrige(x, y):
    # V de Cramer corrige du biais (Bergsma 2013), association cat-cat.
    # Correction utile quand les categories sont desequilibrees (ex: loan_intent).
    tab = pd.crosstab(x, y)
    chi2 = chi2_contingency(tab)[0]
    n = tab.sum().sum()
    phi2 = chi2 / n
    r, k = tab.shape
    # Correction de Bergsma
    phi2corr = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    return np.sqrt(phi2corr / denom) if denom > 0 else 0.0

def correlation_ratio(categories, values):
    # Rapport de correlation eta, association cat-numerique.
    # eta = racine(variance inter-groupes / variance totale).
    categories = np.asarray(categories)
    values = np.asarray(values, dtype=float)
    cats = pd.unique(categories)
    y_mean = values.mean()
    ss_between = 0.0
    for c in cats:
        grp = values[categories == c]
        ss_between += len(grp) * (grp.mean() - y_mean) ** 2
    ss_total = ((values - y_mean) ** 2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else 0.0

def association_matrix(data, num_vars, cat_vars):
    # Matrice association mixte : route selon le couple de types.
    # Spearman / V de Cramer / eta. Valeurs en magnitude 0-1.
    cols = num_vars + cat_vars
    M = pd.DataFrame(np.zeros((len(cols), len(cols))), index=cols, columns=cols)
    for a in cols:
        for b in cols:
            if a == b:
                M.loc[a, b] = 1.0
            elif a in num_vars and b in num_vars:
                M.loc[a, b] = abs(spearmanr(data[a], data[b]).correlation)
            elif a in cat_vars and b in cat_vars:
                M.loc[a, b] = cramers_v_corrige(data[a], data[b])
            else:  # un num, un cat
                cat, num = (a, b) if a in cat_vars else (b, a)
                M.loc[a, b] = correlation_ratio(data[cat], data[num])
    return M.round(3)

print("✓ Fonctions d'association typées définies (Spearman / V de Cramér Bergsma / η)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 3.3 — Matrice d'association mixte unifiée + heatmap
# ═══════════════════════════════════════════════════════════════════════════

assoc = association_matrix(df, NUM_VARS, CAT_VARS)

fig_assoc = px.imshow(
    assoc, text_auto=True, zmin=0, zmax=1,
    color_continuous_scale=[[0, NAVY], [0.5, DEEP], [1, MINT]],
    title="Matrice d'association mixte — mesure routée selon le type (magnitude 0-1)"
)
fig_assoc.update_layout(
    paper_bgcolor=NAVY, plot_bgcolor=NAVY,
    font_color='#C8DDE8', title_font_color=MINT, height=650
)
fig_assoc.show()

print("Lecture : num<->num = Spearman | cat<->cat = V de Cramér (Bergsma) | num<->cat = η")
print("Toutes les valeurs sont en magnitude [0,1] : 0 = pas d'association, 1 = association parfaite.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 3.4 — Association avec la cible loan_status (classement)
# ═══════════════════════════════════════════════════════════════════════════

assoc_cible = {}
# Numériques vs cible binaire -> η (la cible joue le rôle de "catégorie" 0/1)
for v in NUM_VARS:
    assoc_cible[v] = correlation_ratio(df[TARGET], df[v])
# Catégorielles vs cible binaire -> V de Cramér
for v in CAT_VARS:
    assoc_cible[v] = cramers_v_corrige(df[v], df[TARGET])

s_cible = pd.Series(assoc_cible).sort_values(ascending=True)

fig_cible = px.bar(
    x=s_cible.values, y=s_cible.index, orientation='h',
    color=s_cible.values, color_continuous_scale=[[0, DEEP], [1, MINT]],
    title="Association de chaque variable avec le défaut (loan_status)",
    labels={'x': 'Force d\'association (0-1)', 'y': ''}
)
fig_cible.update_layout(
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    coloraxis_showscale=False, height=420
)
fig_cible.show()

print("Classement des variables par force d'association au défaut :")
for v, val in s_cible.sort_values(ascending=False).items():
    mesure = 'η   ' if v in NUM_VARS else 'V de Cramér'
    print(f"  {v:30s} {val:.3f}  ({mesure})")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 3.5 — VIF : diagnostic de multicolinéarité (bloc numérique uniquement)
# ═══════════════════════════════════════════════════════════════════════════
# Le VIF ne concerne QUE les variables explicatives entre elles (la cible n'intervient pas).
# On régresse chaque variable numérique sur les autres : VIF = 1 / (1 - R²).
# Fonction canonique statsmodels : variance_inflation_factor.
# ATTENTION : elle exige une constante ajoutée (add_constant), sinon VIF gonflés.

X_vif = df[NUM_VARS].copy()
X_vif_const = add_constant(X_vif)  # indispensable pour statsmodels

vif_data = pd.DataFrame({
    'variable': X_vif_const.columns,
    'VIF': [variance_inflation_factor(X_vif_const.values, i)
            for i in range(X_vif_const.shape[1])]
})
# On retire la ligne 'const' (non interprétable comme variable métier)
vif_data = vif_data[vif_data['variable'] != 'const'].sort_values('VIF', ascending=False)

print("VIF (Variance Inflation Factor) — variables numériques :")
print(vif_data.round(2).to_string(index=False))
print("\n  Seuil d'alerte : VIF > 5 (multicolinéarité modérée), VIF > 10 (forte).")
print("  Rappel : debt_service_rate = loan_int_rate × loan_percent_income")
print("  -> il est construit à partir d'elles, une redondance est donc attendue.")


---
## Bloc 4 — Sélection de variables (le cœur de la session)

On dispose de nombreuses variables candidates. Toutes ne sont pas utiles : certaines sont redondantes (vu au VIF), d'autres apportent peu. **Sélectionner** = ne garder que les variables qui améliorent réellement le modèle. Bénéfices : modèle plus **simple**, plus **robuste**, plus **facile à expliquer** au régulateur.

Quatre méthodes, toutes en **scikit-learn** :

1. **Forward selection** — on part de **zéro** variable, on **ajoute** à chaque étape celle qui améliore le plus le score. On s'arrête quand on a atteint le nombre voulu.
2. **Backward elimination** — on part de **toutes** les variables, on **retire** à chaque étape la moins utile.
3. **RFE (Recursive Feature Elimination)** — on entraîne le modèle, on retire la variable la moins importante, on recommence.
4. **Lasso (pénalité L1)** — une régression logistique qui **force à zéro** les coefficients des variables inutiles : celles qui survivent sont sélectionnées.

### ⚙️ Deux réglages de vitesse — CHOIX RETENU

La sélection entraîne beaucoup de modèles. On propose deux configurations :

- **Option A — rigoureuse** : validation croisée **5-fold**, la plus fiable, mais **lente** (plusieurs minutes en salle).
- **Option B — pédagogique** : validation croisée **3-fold** + estimateur léger, tourne en **~1 minute**.

> **✅ Choix retenu pour cette formation : Option B (3-fold, rapide).**
> En salle, un bloc qui tourne 5 minutes casse le rythme. La méthodologie est **identique** ; seul le nombre de folds change, ce qui n'affecte pas les enseignements. Pour un livrable de production, basculez `MODE_SELECTION = 'A'`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 4.0 — Réglage vitesse + HOLDOUT SET + préparation données de sélection
# ═══════════════════════════════════════════════════════════════════════════
# [AUDIT TUFFERY #4] — Ajout d'un holdout set (20 %) pour validation finale.
# Tuffery sépare explicitement apprentissage / test pour détecter le sur-apprentissage.
# La sélection de variables et le scorecard s'entraînent sur le TRAIN uniquement.
# ═══════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import train_test_split

MODE_SELECTION = 'B'

if MODE_SELECTION == 'A':
    N_FOLDS = 5
    print("Mode A (rigoureux) : CV 5-fold — plus fiable, plus lent.")
else:
    N_FOLDS = 3
    print("Mode B (pédagogique) : CV 3-fold — rapide (~1 min). [CHOIX RETENU]")

cv_sel = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

FEATURES_OFFICIELLES = [
    'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate',
    'loan_percent_income', 'cb_person_cred_hist_length',
    'home_RENT', 'home_MORTGAGE', 'home_OWN', 'default_enc',
    'debt_service_rate', 'monthly_payment_proxy', 'log_income', 'high_risk_intent',
]
missing = [f for f in FEATURES_OFFICIELLES if f not in df.columns]
if missing:
    raise ValueError(f"Features manquantes : {missing}")

X_all = df[FEATURES_OFFICIELLES].copy()
y     = df['loan_status'].copy()

# ── Holdout split (stratifié) ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"\nHoldout split (stratifié) :")
print(f"  Train : {X_train.shape[0]:,} lignes  (taux défaut : {y_train.mean()*100:.1f} %)")
print(f"  Test  : {X_test.shape[0]:,} lignes  (taux défaut : {y_test.mean()*100:.1f} %)")

# Standardisation — fit sur train uniquement (pas de data leakage)
scaler_sel = StandardScaler()
X_train_scaled = pd.DataFrame(scaler_sel.fit_transform(X_train),
                               columns=FEATURES_OFFICIELLES, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler_sel.transform(X_test),
                               columns=FEATURES_OFFICIELLES, index=X_test.index)

N_FEATURES_CIBLE = 8
print(f"\nVariables candidates : {len(FEATURES_OFFICIELLES)}")
print(f"Objectif de sélection : {N_FEATURES_CIBLE} variables")
print("\n✓ Holdout set créé — sélection et entraînement sur X_train uniquement")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 4.1 — Forward selection (SequentialFeatureSelector)
# ═══════════════════════════════════════════════════════════════════════════

estimateur_sel = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

sfs_forward = SequentialFeatureSelector(
    estimateur_sel, n_features_to_select=N_FEATURES_CIBLE,
    direction='forward', scoring='roc_auc', cv=cv_sel, n_jobs=-1
)
sfs_forward.fit(X_train_scaled, y_train)
feat_forward = list(X_train_scaled.columns[sfs_forward.get_support()])

print(f"FORWARD — {len(feat_forward)} variables sélectionnées :")
for f in feat_forward:
    print(f"  + {f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 4.2 — Backward elimination (SequentialFeatureSelector)
# ═══════════════════════════════════════════════════════════════════════════

sfs_backward = SequentialFeatureSelector(
    estimateur_sel, n_features_to_select=N_FEATURES_CIBLE,
    direction='backward', scoring='roc_auc', cv=cv_sel, n_jobs=-1
)
sfs_backward.fit(X_train_scaled, y_train)
feat_backward = list(X_train_scaled.columns[sfs_backward.get_support()])

print(f"BACKWARD — {len(feat_backward)} variables sélectionnées :")
for f in feat_backward:
    print(f"  + {f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 4.3 — RFE (Recursive Feature Elimination)
# ═══════════════════════════════════════════════════════════════════════════

rfe = RFE(estimateur_sel, n_features_to_select=N_FEATURES_CIBLE)
rfe.fit(X_train_scaled, y_train)
feat_rfe = list(X_train_scaled.columns[rfe.get_support()])

print(f"RFE — {len(feat_rfe)} variables sélectionnées :")
for f in feat_rfe:
    print(f"  + {f}")

# Classement RFE (rang 1 = gardé, rangs suivants = ordre d'élimination)
rang = pd.DataFrame({'variable': FEATURES_OFFICIELLES, 'rang_RFE': rfe.ranking_})
print("\nOrdre d'élimination RFE (rang 1 = conservé) :")
print(rang.sort_values('rang_RFE').to_string(index=False))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 4.4 — Lasso (régression logistique pénalité L1)
# ═══════════════════════════════════════════════════════════════════════════
# Le Lasso met à zéro les coefficients des variables inutiles.
# C = inverse de la force de régularisation : plus C est petit, plus on élague.

lasso_lr = LogisticRegression(
    penalty='l1', solver='liblinear', C=0.02,
    max_iter=1000, random_state=RANDOM_STATE
)
lasso_lr.fit(X_train_scaled, y_train)

coefs = pd.Series(lasso_lr.coef_[0], index=FEATURES_OFFICIELLES)
feat_lasso = list(coefs[coefs.abs() > 1e-6].index)

print(f"LASSO (C=0.02) — {len(feat_lasso)} variables retenues (coef != 0) :")
for f in feat_lasso:
    print(f"  + {f:30s} coef = {coefs[f]:+.3f}")

elimine = [f for f in FEATURES_OFFICIELLES if f not in feat_lasso]
if elimine:
    print(f"\n  Éliminées (coef = 0) : {elimine}")


---
## Bloc 5 — Comparaison des jeux de variables sélectionnés

Chaque méthode a proposé un sous-ensemble. On les compare sur :
- le **nombre de variables** (parcimonie),
- l'**AUC en validation croisée** (performance),
- les **variables communes** (robustesse du signal).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 5 — Comparaison AUC des jeux sélectionnés
# ═══════════════════════════════════════════════════════════════════════════

def auc_cv(features, X_scaled, y, cv):
    # AUC moyenne en CV pour un LR sur un sous-ensemble de features.
    est = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    scores = cross_val_score(est, X_scaled[features], y, scoring='roc_auc', cv=cv, n_jobs=-1)
    return scores.mean()

jeux = {
    'Toutes (14)': FEATURES_OFFICIELLES,
    'Forward':     feat_forward,
    'Backward':    feat_backward,
    'RFE':         feat_rfe,
    'Lasso':       feat_lasso,
}

compare = []
for nom, feats in jeux.items():
    compare.append({
        'Méthode': nom,
        'Nb variables': len(feats),
        'AUC (CV)': round(auc_cv(feats, X_train_scaled, y_train, cv_sel), 4),
    })
df_compare = pd.DataFrame(compare).sort_values('AUC (CV)', ascending=False)
print("Comparaison des jeux de variables (LR, CV) :")
print(df_compare.to_string(index=False))

# Variables communes à forward, backward et RFE
communes = set(feat_forward) & set(feat_backward) & set(feat_rfe)
print(f"\nVariables communes (forward ∩ backward ∩ RFE) : {sorted(communes)}")

# Visualisation : AUC vs parcimonie
fig_cmp = px.scatter(
    df_compare, x='Nb variables', y='AUC (CV)', text='Méthode',
    color='AUC (CV)', color_continuous_scale=[[0, DEEP], [1, MINT]],
    title="Parcimonie vs performance — chaque méthode de sélection"
)
fig_cmp.update_traces(textposition='top center', marker=dict(size=14))
fig_cmp.update_layout(
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    coloraxis_showscale=False, height=430
)
fig_cmp.show()

print("\nLecture : un point en haut à gauche = peu de variables ET bonne AUC = idéal.")


---
## Bloc 6 — Modèle de scoring final : scorecard logistique

On construit le **scorecard** — le modèle interprétable de référence — sur un jeu de variables sélectionné. Chaque coefficient devient un **odds-ratio** :

$$ \text{odds-ratio} = e^{\text{coefficient}} $$

- odds-ratio **> 1** : la variable **augmente** le risque de défaut.
- odds-ratio **< 1** : la variable **protège** (diminue le risque).

> **Rappel — rôle du scorecard :** c'est le **benchmark interprétable/réglementaire**, à côté du RF de production. Il **n'explique pas** le RF ; il est le modèle transparent qu'on pourrait déployer si le régulateur exige une transparence maximale.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 6 — Scorecard LR sur jeu sélectionné + odds-ratios
# ═══════════════════════════════════════════════════════════════════════════

# On retient le jeu "backward" comme scorecard parcimonieux (modifiable).
# (choix pédagogique : souvent proche de forward/RFE, bon compromis parcimonie/AUC)
FEATURES_SCORECARD = feat_backward
print(f"Scorecard construit sur {len(FEATURES_SCORECARD)} variables : {FEATURES_SCORECARD}\n")

lr_scorecard = LogisticRegression(max_iter=1000, class_weight='balanced',
                                   random_state=RANDOM_STATE)
lr_scorecard.fit(X_train_scaled[FEATURES_SCORECARD], y_train)

scorecard = pd.DataFrame({
    'variable': FEATURES_SCORECARD,
    'coefficient': lr_scorecard.coef_[0],
    'odds_ratio': np.exp(lr_scorecard.coef_[0]),
}).sort_values('odds_ratio', ascending=False)
scorecard['effet'] = np.where(scorecard['odds_ratio'] > 1, 'augmente le risque', 'protège')

print("SCORECARD — coefficients & odds-ratios (variables standardisées) :")
print(scorecard.round(3).to_string(index=False))

fig_sc = px.bar(
    scorecard.sort_values('odds_ratio'),
    x='odds_ratio', y='variable', orientation='h',
    color='odds_ratio', color_continuous_scale=[[0, DEEP], [0.5, TEAL], [1, ORANGE]],
    title='Scorecard — odds-ratios par variable (>1 = risque, <1 = protection)'
)
fig_sc.add_vline(x=1.0, line_dash='dash', line_color='white')
fig_sc.update_layout(
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    coloraxis_showscale=False, height=420
)
fig_sc.show()

print("\n> Ce scorecard est un livrable pédagogique interprétable, à côté du parquet de scores.")
print("> Il ne remplace pas lr_shadow_score (calculé, lui, sur les 14 features — cf. Bloc 9).")


---
## Bloc 6bis — Grille de score en points (0 – 100) [AUDIT TUFFERY #2]

> **Audit Tuffery — Lacune #2 (importante)**

Tuffery ne s'arrête pas aux odds-ratios. Il convertit chaque coefficient en **points entiers**
pour produire une grille auditable ligne à ligne par un analyste crédit ou un régulateur.

Formule de conversion (méthode PDO — Points to Double the Odds) :
- On fixe une **plage cible** (ici 0-100 points)
- On mappe linéairement les coefficients standardisés dans cette plage
- On arrondit à l'entier le plus proche

La grille finale donne, pour chaque variable, le nombre de points attribués à chaque modalité.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 6bis — Conversion scorecard → grille de points entiers (0-100)
# ═══════════════════════════════════════════════════════════════════════════
# Méthode : mapping linéaire des coefficients dans une plage cible [0, SCORE_MAX].
# Le coefficient le plus négatif → 0 pt, le plus positif → SCORE_MAX pts.
# Tuffery utilise cette approche (section 18.8) pour produire une grille auditable.
# ═══════════════════════════════════════════════════════════════════════════

SCORE_MAX = 100  # plage cible : 0-100 points (modifiable : 300-850 style FICO)

coefs = pd.Series(lr_scorecard.coef_[0], index=FEATURES_SCORECARD)

# Mapping linéaire min→0, max→SCORE_MAX
c_min, c_max = coefs.min(), coefs.max()

def coef_to_points(c, c_min, c_max, score_max):
    if c_max == c_min:
        return score_max // 2
    return int(round((c - c_min) / (c_max - c_min) * score_max))

grille_points = pd.DataFrame({
    'variable':    FEATURES_SCORECARD,
    'coefficient': coefs.values,
    'odds_ratio':  np.exp(coefs.values),
    'points':      [coef_to_points(c, c_min, c_max, SCORE_MAX) for c in coefs.values],
    'effet':       ['↑ Risque' if c > 0 else '↓ Protège' for c in coefs.values],
}).sort_values('points', ascending=False).reset_index(drop=True)

print("=" * 60)
print(f"GRILLE DE SCORE — {len(FEATURES_SCORECARD)} variables · plage 0-{SCORE_MAX} pts")
print("=" * 60)
print(f"{'Variable':<30} {'Points':>6}  {'Odds-ratio':>10}  {'Effet'}")
print("-" * 60)
for _, row in grille_points.iterrows():
    print(f"{row['variable']:<30} {row['points']:>6}  {row['odds_ratio']:>10.3f}  {row['effet']}")
print("=" * 60)
print(f"  Score minimum atteignable : 0  (profil le moins risqué)")
print(f"  Score maximum atteignable : {SCORE_MAX}  (profil le plus risqué)")

# Visualisation barres horizontales
fig_pts = px.bar(
    grille_points.sort_values('points'),
    x='points', y='variable', orientation='h',
    color='points',
    color_continuous_scale=[[0, MINT], [0.5, TEAL], [1, ORANGE]],
    text='points',
    title=f'Grille de score — points par variable (0 = protège, {SCORE_MAX} = risque max)',
    labels={'points': 'Points (0-100)', 'variable': ''}
)
fig_pts.update_traces(textposition='outside')
fig_pts.update_layout(
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    coloraxis_showscale=False, height=420
)
fig_pts.show()

print("\n> Cette grille est auditable ligne à ligne (conforme Tuffery §18.8).")
print("> Pour appliquer le score à un nouveau dossier : sommer les points des variables présentes.")


---
## Bloc 7 — Random Forest de production & stratégies de déséquilibre

Le scorecard est notre benchmark transparent. Le **modèle de production**, lui, vise la **performance** : c'est un **Random Forest** sur les **14 features complètes**.

On compare 3 familles (LR / RF / GB) × 3 stratégies de déséquilibre (Baseline / `class_weight='balanced'` / SMOTE-NC) pour :
- confirmer que **RF Baseline est champion** (AUC ≈ 0,929),
- vérifier que **SMOTE-NC dégrade** (ratio de déséquilibre 3,6:1, modéré).

> **Anti-leakage :** SMOTE-NC est appliqué **dans** chaque fold, jamais avant le split.
> **Important :** le RF de production garde **toutes** ses 14 features — la sélection du Bloc 4 s'applique au scorecard, pas au RF.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 7 — CV comparative (RF de production sur 14 features)
# ═══════════════════════════════════════════════════════════════════════════

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X = df[FEATURES_OFFICIELLES].copy()  # non standardisé : RF/GB n'en ont pas besoin

cat_cols_idx = [FEATURES_OFFICIELLES.index(c) for c in
                ['home_RENT', 'home_MORTGAGE', 'home_OWN', 'default_enc', 'high_risk_intent']]

def evaluate_cv(model, X, y, smote=False):
    aucs, f1s, recalls, precisions = [], [], [], []
    for tr, te in cv.split(X, y):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        if smote:
            sm = SMOTENC(categorical_features=cat_cols_idx, random_state=RANDOM_STATE)
            Xtr, ytr = sm.fit_resample(Xtr, ytr)
        model.fit(Xtr, ytr)
        proba = model.predict_proba(Xte)[:, 1]
        pred = model.predict(Xte)
        aucs.append(roc_auc_score(yte, proba)); f1s.append(f1_score(yte, pred))
        recalls.append(recall_score(yte, pred)); precisions.append(precision_score(yte, pred))
    return {'AUC': np.mean(aucs), 'F1': np.mean(f1s),
            'Recall': np.mean(recalls), 'Precision': np.mean(precisions)}

configs = []
print("Évaluation des configurations (CV 5-fold)...\n")

for nom, mdl, sm in [
    ('LR', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), False),
    ('LR-bal', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE), False),
    ('RF', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1), False),
    ('RF-bal', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1), False),
    ('RF-SMOTE', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1), True),
    ('GB', GradientBoostingClassifier(random_state=RANDOM_STATE), False),
]:
    r = evaluate_cv(mdl, X, y, smote=sm)
    r['Config'] = nom
    configs.append(r)
    print(f"  ✓ {nom:10s} AUC={r['AUC']:.3f}")

df_results = pd.DataFrame(configs)[['Config', 'AUC', 'F1', 'Recall', 'Precision']] \
    .sort_values('AUC', ascending=False).reset_index(drop=True)
print("\nClassement :")
print(df_results.round(3).to_string(index=False))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 7bis — Entraînement final du champion RF + shadow LR
# ═══════════════════════════════════════════════════════════════════════════

# RF de production : 14 features, entraîné sur tout le dataset
rf_final = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf_final.fit(X, y)

# Shadow LR : calculé sur les 14 MÊMES features (cohérence aval J2S3/J3S1/J3S2)
lr_shadow = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
lr_shadow.fit(X, y)

print("✓ RF Baseline entraîné sur 14 features (modèle de production)")
print("✓ LR Balanced (shadow) entraîné sur les 14 mêmes features")

# Feature importance RF
importances = pd.DataFrame({
    'feature': FEATURES_OFFICIELLES, 'importance': rf_final.feature_importances_
}).sort_values('importance', ascending=True)

fig_imp = px.bar(importances, x='importance', y='feature', orientation='h',
                 color='importance', color_continuous_scale=[[0, DEEP], [1, MINT]],
                 title='Feature Importance — RF de production (14 features)')
fig_imp.update_layout(paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
                      font_color='#C8DDE8', title_font_color=MINT,
                      coloraxis_showscale=False, height=450)
fig_imp.show()


---
## Bloc 7ter — Courbe ROC comparative [AUDIT TUFFERY #5]

> **Audit Tuffery — Recommandation #5**

Tuffery trace la courbe ROC des deux modèles (Figure 18.8) pour visualiser leurs zones
de force respective. Ici on compare RF vs LR sur le **holdout set** (données jamais vues).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 7ter — Courbe ROC sur holdout set (RF vs LR)
# ═══════════════════════════════════════════════════════════════════════════

from sklearn.metrics import roc_curve, auc as sklearn_auc

# Scores sur le holdout set
proba_rf_test = rf_final.predict_proba(X_test)[:, 1]
proba_lr_test = lr_shadow.predict_proba(X_test)[:, 1]

fpr_rf, tpr_rf, _ = roc_curve(y_test, proba_rf_test)
fpr_lr, tpr_lr, _ = roc_curve(y_test, proba_lr_test)

auc_rf_test = sklearn_auc(fpr_rf, tpr_rf)
auc_lr_test = sklearn_auc(fpr_lr, tpr_lr)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=fpr_rf, y=tpr_rf, mode='lines',
    line=dict(color=MINT, width=2.5),
    name=f'Random Forest  (AUC = {auc_rf_test:.3f})'))
fig_roc.add_trace(go.Scatter(x=fpr_lr, y=tpr_lr, mode='lines',
    line=dict(color=ORANGE, width=2.5, dash='dash'),
    name=f'Régression Log. (AUC = {auc_lr_test:.3f})'))
fig_roc.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
    line=dict(color='grey', width=1, dash='dot'), name='Aléatoire'))

fig_roc.update_layout(
    title='Courbe ROC — Holdout set (20 % jamais vus pendant l'entraînement)',
    xaxis_title='Taux de faux positifs (FPR)',
    yaxis_title='Taux de vrais positifs (TPR)',
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT, height=450,
    legend=dict(x=0.6, y=0.1, bgcolor='rgba(0,0,0,0.3)')
)
fig_roc.show()

print(f"AUC sur holdout set (données jamais vues) :")
print(f"  Random Forest      : {auc_rf_test:.3f}")
print(f"  Régression Log.    : {auc_lr_test:.3f}")
print(f"  AUC RF en CV train : ~0.929  (référence)")
delta = 0.929 - auc_rf_test
if delta > 0.02:
    print(f"\n⚠ Sur-apprentissage détecté : delta = {delta:.3f} (> 0.02)")
else:
    print(f"\n✓ Pas de sur-apprentissage significatif : delta = {delta:.3f}")


---
## Bloc 8 — Seuil de décision optimal

Logique métier : un **Faux Négatif** (défaut non détecté) coûte 5 à 10× plus qu'un **Faux Positif** (bon dossier refusé). On documente le seuil retenu, **t = 0,42**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 8 — Recherche du seuil optimal
# ═══════════════════════════════════════════════════════════════════════════

proba_full = rf_final.predict_proba(X)[:, 1]
thresholds = np.arange(0.30, 0.55, 0.01)
rows = []
for t in thresholds:
    pred_t = (proba_full >= t).astype(int)
    rows.append({'seuil': round(t, 2), 'f1': f1_score(y, pred_t),
                 'recall': recall_score(y, pred_t), 'precision': precision_score(y, pred_t)})
df_seuils = pd.DataFrame(rows)

SEUIL_PRODUCTION = 0.42
print(f"Seuil retenu en production : t = {SEUIL_PRODUCTION}")
r042 = df_seuils[df_seuils['seuil'] == SEUIL_PRODUCTION]
if not r042.empty:
    print(f"  F1={r042['f1'].values[0]:.3f} | Recall={r042['recall'].values[0]:.3f} | "
          f"Precision={r042['precision'].values[0]:.3f}")

fig_s = go.Figure()
for col, color, name in [('f1', MINT, 'F1'), ('recall', TEAL, 'Recall'), ('precision', ORANGE, 'Precision')]:
    fig_s.add_trace(go.Scatter(x=df_seuils['seuil'], y=df_seuils[col], mode='lines',
                               line=dict(color=color, width=2.5), name=name))
fig_s.add_vline(x=SEUIL_PRODUCTION, line_dash='dash', line_color='white',
                annotation_text=f't={SEUIL_PRODUCTION}', annotation_font_color='white')
fig_s.update_layout(title='Recherche du seuil optimal', xaxis_title='Seuil t', yaxis_title='Score',
                    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
                    font_color='#C8DDE8', title_font_color=MINT, height=400)
fig_s.show()


---
## Bloc 8bis — Découpage en tranches de risque [AUDIT TUFFERY #3]

> **Audit Tuffery — Lacune #3 (importante)**

Tuffery ne s'arrête pas à un seuil unique. Il définit **3 tranches de risque** à partir des déciles,
avec pour chaque tranche les effectifs et le taux d'impayé observé.

Ces tranches permettent de moduler la politique de crédit :
- **Risque faible** → instruction automatique (rapide, peu de vérifications)
- **Risque moyen** → instruction normale (examen attentif)
- **Risque fort** → instruction hiérarchique (directeur d'agence)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 8bis — Découpage en tranches de risque (déciles → 3 tranches)
# ═══════════════════════════════════════════════════════════════════════════
# Méthode Tuffery (section 18.8) : on passe par les déciles pour identifier
# des seuils naturels, puis on définit 3 tranches opérationnelles.
# ═══════════════════════════════════════════════════════════════════════════

# Calcul des déciles sur l'ensemble complet
df['rf_score_proba'] = rf_final.predict_proba(X)[:, 1]
df['decile'] = pd.qcut(df['rf_score_proba'], q=10, labels=False, duplicates='drop')

# Taux de défaut par décile
decile_stats = df.groupby('decile').agg(
    N=('loan_status', 'count'),
    n_defaut=('loan_status', 'sum'),
    score_min=('rf_score_proba', 'min'),
    score_max=('rf_score_proba', 'max'),
).reset_index()
decile_stats['taux_defaut_pct'] = (decile_stats['n_defaut'] / decile_stats['N'] * 100).round(1)

print("Taux de défaut par décile de score :")
print(f"{'Décile':>7} {'N':>6} {'Défauts':>8} {'Taux (%)':>9} {'Score min':>10} {'Score max':>10}")
print("-" * 55)
for _, r in decile_stats.iterrows():
    print(f"  {int(r['decile']):>3}     {r['N']:>6} {r['n_defaut']:>8} {r['taux_defaut_pct']:>8.1f}%"
          f"  [{r['score_min']:.3f} – {r['score_max']:.3f}]")

# Identification des seuils naturels (entre décile 3→4 et 7→8, typiquement)
seuil_faible = decile_stats.loc[decile_stats['decile'] == 3, 'score_max'].values[0]
seuil_fort   = decile_stats.loc[decile_stats['decile'] == 7, 'score_max'].values[0]
print(f"\nSeuils retenus : faible < {seuil_faible:.3f} ≤ moyen < {seuil_fort:.3f} ≤ fort")

# Application des tranches
def tranche(p):
    if p < seuil_faible:  return 'A — Risque faible'
    elif p < seuil_fort:  return 'B — Risque moyen'
    else:                 return 'C — Risque fort'

df['tranche_risque'] = df['rf_score_proba'].apply(tranche)

# Statistiques par tranche
tranches = df.groupby('tranche_risque').agg(
    N=('loan_status', 'count'),
    n_defaut=('loan_status', 'sum'),
    pct_population=('loan_status', lambda x: len(x) / len(df) * 100),
).reset_index()
tranches['taux_defaut_pct'] = (tranches['n_defaut'] / tranches['N'] * 100).round(1)
tranches['pct_population']  = tranches['pct_population'].round(1)

print("\n" + "=" * 65)
print("TABLEAU DE DÉCISION — 3 tranches de risque")
print("=" * 65)
print(f"{'Tranche':<22} {'N':>6} {'% Pop.':>7} {'Défauts':>8} {'Taux (%)':>9}  {'Instruction'}")
print("-" * 65)
instructions = {
    'A — Risque faible': 'Automatique (rapide)',
    'B — Risque moyen':  'Examen attentif',
    'C — Risque fort':   'Hiérarchique (directeur)',
}
for _, r in tranches.iterrows():
    print(f"  {r['tranche_risque']:<20} {r['N']:>6} {r['pct_population']:>6.1f}%"
          f" {r['n_defaut']:>8} {r['taux_defaut_pct']:>8.1f}%  → {instructions[r['tranche_risque']]}")
print("=" * 65)

# Visualisation
fig_tr = px.bar(
    tranches, x='tranche_risque', y='taux_defaut_pct',
    color='tranche_risque',
    color_discrete_map={
        'A — Risque faible': MINT, 'B — Risque moyen': TEAL, 'C — Risque fort': ORANGE},
    text='taux_defaut_pct',
    title='Taux de défaut par tranche de risque',
    labels={'taux_defaut_pct': 'Taux de défaut (%)', 'tranche_risque': ''}
)
fig_tr.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_tr.update_layout(
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    showlegend=False, height=400
)
fig_tr.show()

print("\n✓ Tranches de risque créées — conforme Tuffery §18.8")
print(f"  'tranche_risque' ajoutée au DataFrame ({df['tranche_risque'].value_counts().to_dict()})")


---
## Bloc 9 — Production des scores & Parquet final

Livrable : `credit_risk_scored.parquet` = entrée de J2S3 (SHAP). Ajout de 3 colonnes :
`rf_score_proba`, `rf_decision_042`, `lr_shadow_score`.

> **`lr_shadow_score` est calculé sur les 14 features** (identique à l'existant) — garantit zéro impact sur J2S3 / J3S1 / J3S2. Le scorecard parcimonieux du Bloc 6 est un livrable séparé.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 9 — Production des scores & sauvegarde
# ═══════════════════════════════════════════════════════════════════════════

df['rf_score_proba']  = rf_final.predict_proba(X)[:, 1]
df['rf_decision_042'] = (df['rf_score_proba'] >= SEUIL_PRODUCTION).astype(int)
df['lr_shadow_score'] = lr_shadow.predict_proba(X)[:, 1]   # 14 features, cohérence aval

df.to_parquet(PARQUET_OUT, index=False)
print(f"✓ Fichier sauvegardé : {PARQUET_OUT}")

# Relecture & assertions
df_check = pd.read_parquet(PARQUET_OUT)
assert 'rf_score_proba'  in df_check.columns, "ERREUR : rf_score_proba absent"
assert 'rf_decision_042' in df_check.columns, "ERREUR : rf_decision_042 absent"
assert 'lr_shadow_score' in df_check.columns, "ERREUR : lr_shadow_score absent"
assert df_check.isnull().sum().sum() == 0, "ERREUR : NaN dans le parquet"
assert 'loan_grade' not in df_check.columns, "ERREUR : loan_grade présente"
assert df_check['rf_score_proba'].between(0, 1).all(), "ERREUR : proba hors [0,1]"
assert abs(df_check['loan_status'].mean()*100 - 21.8) < 0.15, "ERREUR : taux défaut != 21.8%"

auc_final = roc_auc_score(df_check['loan_status'], df_check['rf_score_proba'])
print(f"\n  Shape       : {df_check.shape}")
print(f"  AUC (train) : {auc_final:.3f}  (référence CV out-of-sample : 0,929)")
print(f"  Défaut      : {df_check['loan_status'].mean()*100:.1f} %")
print("\n✓ credit_risk_scored.parquet validé — prêt pour J2S3 SHAP")


---
## Bloc 10 — Commit Git

```bash
git add data/processed/credit_risk_scored.parquet
git commit -m "feat(j2s2): scoring credit & selection de variables — credit_risk_scored.parquet"
git push origin main
```

> **Google Colab** : préfixer chaque commande avec `!`

---
## Récapitulatif J2S2 — Ce que vous avez produit

| Étape | Action | Résultat |
|-------|--------|----------|
| **Scoring** | Notion de score, scorecard, odds-ratio | Cadre conceptuel |
| **Bivarié typé** | Spearman / V de Cramér / η + VIF | Associations correctes par type |
| **Sélection** | forward / backward / RFE / Lasso | Jeux de variables comparés |
| **Comparaison** | AUC vs parcimonie | Convergence des méthodes |
| **Scorecard LR** | odds-ratios sur jeu sélectionné | Benchmark interprétable |
| **RF production** | 14 features, CV 5-fold | **AUC = 0,929** |
| **Shadow LR** | 14 features | lr_shadow_score (cohérence aval) |
| **Seuil** | t = 0,42 | Coût FN/FP documenté |
| **Parquet** | credit_risk_scored.parquet | Livrable J2S2 |

### Deux modèles, deux rôles
> **RF** = production (performance), expliqué par **ses propres SHAP** en J2S3.
> **Scorecard LR** = benchmark interprétable/réglementaire — il **n'explique pas** le RF.

### Sur la sélection de variables
> forward / backward / RFE / Lasso convergent souvent vers un noyau commun de variables.
> La sélection s'applique au **scorecard** ; le **RF de production garde ses 14 features**.

> **Demain J2S3 :** Explicabilité SHAP → waterfall par dossier + importance globale, sur le RF de production.

---
## Annexe (optionnel) — Test : `cluster_id` comme feature additionnelle

> ⚠ **Section optionnelle, hors pipeline officiel.** Objectif : vérifier empiriquement si `cluster_id` (J2S1) apporte un signal au-delà des features qui ont servi à le construire. Le résultat **ne modifie pas** le pipeline ni l'AUC=0,929.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ANNEXE — Test exploratoire cluster_id (n'affecte pas le pipeline officiel)
# ═══════════════════════════════════════════════════════════════════════════

if 'cluster_label' in df.columns:
    cluster_dummies = pd.get_dummies(df['cluster_label'], prefix='cluster', dtype=int)
    X_clu = pd.concat([df[FEATURES_OFFICIELLES], cluster_dummies], axis=1)

    rf_clu = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    auc_clu = evaluate_cv(rf_clu, X_clu, y)['AUC']
    auc_off = df_results[df_results['Config'] == 'RF']['AUC'].values[0]

    print(f"RF officiel (sans cluster_id) : AUC = {auc_off:.4f}")
    print(f"RF + cluster_id (one-hot)     : AUC = {auc_clu:.4f}")
    print(f"Delta : {auc_clu - auc_off:+.4f}")
    print("\n> Gain généralement négligeable — cluster_id reste un outil de reporting/segmentation,")
    print("> pas une variable d'entrée du modèle de scoring.")
else:
    print("cluster_label absent — annexe ignorée (pipeline officiel inchangé).")
